# ⚖️ Unified Legal Assistant — Colab (T4) runner

Runs the **unified UI** (one chat, all projects) on a Colab **T4 GPU (16 GB)**.

Topology: Ollama + the RAG microservices + the Gradio UI all run as local
processes inside this one VM. **Neo4j is already cloud-hosted (Aura)**, so no
database runs here — you only need the Aura password.

**Before you start, have ready:**
1. A **GitHub token** (repo is private) — Settings ▸ Developer settings ▸ Personal access tokens (classic), `repo` scope.
2. Your **Neo4j Aura password**.
3. The **494 MB QLoRA knowledge adapter** uploaded to your Google Drive (it is git-ignored, so it does NOT clone). Needed only for the *Finetuned* project.

Run the cells top to bottom. The **last cell** prints a public `*.gradio.live` link.

## 1 · Check the GPU
Make sure **Runtime ▸ Change runtime type ▸ T4 GPU** is selected.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '| bf16 supported:', torch.cuda.is_bf16_supported())
# On T4 bf16 is False -> the app auto-uses fp16 for the 4-bit Qwen. No action needed.

## 2 · Clone the repo (private → token)
The token is read interactively so it isn't saved in the notebook.

In [ ]:
import getpass, os
TOKEN = getpass.getpass('GitHub token (repo scope): ')
REPO = 'ayanasser/LegalPolicy_LLM'
!git clone -b unified-ui https://{TOKEN}@github.com/{REPO}.git /content/LegalPolicy_LLM
%cd /content/LegalPolicy_LLM
del TOKEN  # don't keep it around
!git log --oneline -3

## 3 · Install dependencies
Colab already ships a CUDA build of **torch** — we do NOT reinstall it. This
installs the *core* stack (local HF Qwen + Bilingual RAG + Neo4j-Aura graph).
Takes a few minutes.

In [ ]:
!pip install -q \
  gradio pandas python-dotenv pyyaml requests \
  transformers peft bitsandbytes accelerate sentencepiece einops \
  chromadb sentence-transformers ollama \
  fastapi 'uvicorn[standard]' pydantic pydantic-settings \
  neo4j FlagEmbedding langfuse
print('core deps installed.')

In [ ]:
# OPTIONAL — only if you also want the Multi-Agent (LangGraph) project.
# !pip install -q langchain langchain-community langgraph instructor \
#   claude-agent-sdk lightrag-hku nltk httpx duckduckgo-search

## 4 · Install + start Ollama, pull the chat models
Ollama serves the Llama baseline, prompt-design, and the RAG answer models.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
for m in ['qwen2.5:3b-instruct', 'qwen3:4b', 'llama3.2:3b']:
    print('pulling', m)
    subprocess.run(['ollama','pull',m], check=True)
!ollama list

## 5 · Bring the QLoRA adapter from Google Drive
Needed only for the **Finetuned** project. Edit `ADAPTER_SRC` to your Drive path.
Skip this cell if you don't need the finetuned backend.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ▼▼▼ EDIT THIS to where the adapter folder lives on your Drive ▼▼▼
ADAPTER_SRC = '/content/drive/MyDrive/LegalPolicy/qlora-qwen2.5-3b-knowledge'
!mkdir -p runs && cp -r "{ADAPTER_SRC}" runs/
!ls -la runs/qlora-qwen2.5-3b-knowledge | head

## 6 · Configuration (`.env`)
Put your Aura password here. `BRAG_EMBED_DEVICE=cuda` keeps BGE-M3 on the T4
(saves Colab's limited system RAM). The 4-bit compute dtype auto-detects to fp16
on T4 — no need to set it.

In [ ]:
%%writefile .env
NEO4J_PASSWORD=PUT_YOUR_AURA_PASSWORD_HERE
BRAG_EMBED_DEVICE=cuda
# Optional Langfuse tracing:
# LANGFUSE_PUBLIC_KEY=...
# LANGFUSE_SECRET_KEY=...
# LANGFUSE_HOST=https://cloud.langfuse.com

## 7 · Build the Chroma index (Bilingual RAG)
The 33 MB index is git-ignored, so rebuild it from the tracked corpus
(`data/orig_data.json`). Takes a couple of minutes on the T4. Skip if you copied
a prebuilt `artifacts/bilingual_rag/chroma_db` from Drive instead.

In [ ]:
import os
os.environ['BRAG_EMBED_DEVICE'] = 'cuda'
!python -m apps.bilingual_rag.build_index

## 8 · Start the RAG microservices (background)
Bilingual RAG on :8100 and the Neo4j-Aura graph RAG on :8000. They keep running
as long as this kernel is alive.

In [ ]:
import subprocess, os, time
env = {**os.environ, 'PYTHONPATH': 'src', 'BRAG_EMBED_DEVICE': 'cuda'}

def serve(module, port, log):
    return subprocess.Popen(
        ['python','-m','uvicorn', module, '--host','0.0.0.0','--port',str(port)],
        env=env, stdout=open(log,'w'), stderr=subprocess.STDOUT)

serve('apps.bilingual_rag.api:app', 8100, '/content/bilingual.log')
serve('apps.api.main:app',          8000, '/content/neo4j.log')   # uses Aura
print('starting services… (first start loads BGE-M3, ~30-60s)')
time.sleep(45)
!echo '--- bilingual (:8100) ---'; tail -n 6 /content/bilingual.log
!echo '--- neo4j-aura (:8000) ---'; tail -n 6 /content/neo4j.log
!curl -s localhost:8100/health; echo; curl -s localhost:8000/health; echo

## 9 · Launch the Unified UI (public link)
`LP_SHARE=1` prints a `*.gradio.live` URL. **Keep this cell running** — stopping
it stops the UI. The first message to each project loads its model (a ticking
'⏳ loading…' status shows in the chat); later questions are fast.

In [ ]:
import os
os.environ['PYTHONPATH'] = 'src'
os.environ['LP_SHARE'] = '1'
os.environ['BRAG_EMBED_DEVICE'] = 'cuda'
!python -m apps.unified_ui.app

## Troubleshooting

- **A RAG project says 'service unavailable'** → its cell-8 process didn't start. Check `/content/bilingual.log` or `/content/neo4j.log`.
- **Graph project errors** → wrong/empty `NEO4J_PASSWORD` in `.env`, or Aura instance is paused (free Aura sleeps after 3 idle days — resume it in the Aura console).
- **Finetuned project fails to load** → adapter not copied (step 5) or wrong `ADAPTER_SRC` path.
- **Out of system RAM** → you're on a free (non-high-RAM) VM. Don't run the Multi-Agent + every service at once; embedders are already on the GPU via `BRAG_EMBED_DEVICE=cuda`.
- **Session disconnects** → free Colab caps runtime (~12h) and reclaims idle GPUs. Re-run from the top.
- **Multi-Agent project errors** → run the optional install in step 3.